# CPU re-execution of the 80 full-data runs (Amendment 25)

**Step A (cells 1-5) is a one-hour test.** It re-runs the 4 full-data runs that were already reproduced bit for bit on the pod, now on Colab's CPU, in the exact software of the original runs (Python 3.10, PyTorch 2.4.1, NumPy 1.26.4).
- If it prints **QUICK CHECK PASSED**, run cell 6, which does all 80 runs (likely 2-3 days, in pieces).
- If it prints **FAILED**, stop and send me the output. Colab's processor then gives different last digits from the original machine, and the paper reports 4 of 80 with that reason.

**Before you start:** Runtime → Change runtime type → **CPU**, **High-RAM**. Run cells 1 to 5 in order (not Run all).

In [ ]:
# 1. Drive and folders. Results go straight to Drive, so a disconnect loses at most the run in progress.
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/QML-backup-run'
OUT = f'{D}/cpu17_out'
for sub in ('shards', 'predictions', 'step_norms', 'weights'):
    os.makedirs(f'{OUT}/{sub}/C_17_fulldata', exist_ok=True)
print('already on Drive:', len(os.listdir(f'{OUT}/shards/C_17_fulldata')), 'C_17 runs')

In [ ]:
# 2. Code and the records of the full-data runs (about 10-15 minutes)
R = '/content/repo'
if not os.path.isdir(R):
    !git clone -q "{D}/repo_2026-10-21.bundle" {R}
    !cd {R} && git checkout -q feature/journal-expansion
if not os.path.exists(f'{R}/.cpu17_records'):
    !apt-get -qq install -y pigz > /dev/null
    !tar -I pigz -xf "{D}/replay_backup_2026-10-21.tgz" -C {R} --wildcards '*artifacts/shards/17_fulldata/*' '*artifacts/predictions/17_fulldata/*' '*artifacts/shards/C_17_fulldata/*' '*artifacts/data_cache/*' 2>&1 | tail -2
    !tar -I pigz -xf "{D}/feature_cache_2026-10-10.tgz" -C {R}
    !touch {R}/.cpu17_records
A = f'{R}/artifacts'
# the 4 runs already reproduced on the pod are kept aside, so the test re-runs them
!mkdir -p {A}/pod_C_17 && (mv {A}/shards/C_17_fulldata/* {A}/pod_C_17/ 2>/dev/null; true)
# C_17 outputs live on Drive
for sub in ('shards', 'predictions', 'step_norms', 'weights'):
    p = f'{A}/{sub}/C_17_fulldata'
    os.makedirs(f'{A}/{sub}', exist_ok=True)
    if os.path.islink(p): os.unlink(p)
    elif os.path.isdir(p): os.rmdir(p)
    os.symlink(f'{OUT}/{sub}/C_17_fulldata', p)
!ls {A}/shards/17_fulldata | wc -l; ls {A}/pod_C_17 | wc -l; ls {A}/feature_cache | head -3

In [ ]:
# 3. The original software, in its own Python 3.10 environment (about 10-15 minutes)
if not os.path.exists('/content/env310/bin/python'):
    !cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/env310 -c conda-forge python=3.10 pip > /dev/null
    !/content/env310/bin/pip -q install -r {R}/requirements.lock.txt --extra-index-url https://download.pytorch.org/whl/cu118
!/content/env310/bin/python -c "import sys, torch, numpy, pennylane, sklearn; print(sys.version.split()[0], torch.__version__, numpy.__version__, pennylane.__version__, sklearn.__version__, 'threads', torch.get_num_threads())"
!lscpu | grep -E 'Model name|^CPU\(s\)'

In [ ]:
# 4. Settings shared by the test and the full run
import json, glob
rec = json.load(open(sorted(glob.glob(f'{A}/shards/17_fulldata/*.json'))[0]))
SHA = rec['git_sha']
os.environ['QML_ARTIFACT_ROOT'] = A
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['MPLBACKEND'] = 'Agg'
print('recorded commit', SHA)

In [ ]:
# 5. THE TEST: the 4 runs reproduced on the pod, re-run here. Look for QUICK CHECK PASSED / FAILED at the end.
!cd {R} && /content/env310/bin/python scripts/replay.py --source 17_fulldata --code-at {SHA} --quick 2>&1 | tee "{OUT}/pilot.log" | grep -v "^Using downloaded" | tail -15

**Only if cell 5 printed QUICK CHECK PASSED:** run cell 6. It runs all 80, one at a time, as the original did. Each finished run is saved on Drive. If Colab disconnects, run cells 1 to 4 and then cell 6 again: it skips every run already saved.

In [ ]:
# 6. ALL 80 RUNS (long: likely 2-3 days in pieces). Progress: one line per run.
!cd {R} && /content/env310/bin/python scripts/replay.py --source 17_fulldata --code-at {SHA} 2>&1 | tee -a "{OUT}/full.log" | grep -E "^\[|summary"
